![Platform+](images/banner.png)

# Lab 8: Time Travel
## Undo a bad delete — recover your data safely

> **Persona track: Data Analyst** — same lab, your angle.

:point_right: **In this lesson, you'll build a working analysis table, watch an incorrect `DELETE` wipe real rows out of it, then recover every row with Time Travel — no backup to restore, no support ticket, no admin needed.**

The scenario: You're an analyst at **Roastery & Co.** You've copied the `ORDERS` table into your own sandbox to build a revenue analysis table. Mid-cleanup, you run a `DELETE` to strip out a few test rows — but the `WHERE` clause is incorrect and it removes a busy store's real orders. You notice a minute later. The rows are gone from your table. This lab shows you how to get them back.

We'll cover:
1. What Time Travel is and what it protects you from
2. Building your analysis table (your own copy of `ORDERS`)
3. Reproducing the incident — a bad `DELETE` (and capturing its query id)
4. Looking at a table's contents as they were at an earlier point, with `AT` and `BEFORE`
5. Recovering with a zero-copy point-in-time `CLONE`, then swapping it in for the damaged original
6. Using CoCo as your recovery partner

:books: **By the end you'll be able to:**
- Explain the Time Travel retention window and what it covers
- Query a table's past state with `AT(timestamp => ...)` and `BEFORE(statement => ...)`
- Recover an older state with a zero-copy `CLONE ... BEFORE(...)`
- Verify a recovery *before* you overwrite the live table
- Know when to reach for `UNDROP` instead
- Use CoCo to diagnose and recover from data-loss incidents

**Time:** ~15 min · **Lab interface:** Workspace Notebook with Cortex Code (CoCo) as your tutor.
*Source data lives in the shared, read-only `PLATPLUS_DB`; everything you create lives in your own database, warehouse, and role — each named after your username (e.g. `USER_DB` / `USER_WH` / `USER_RL`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


## Set Your Session Context

Every Snowflake session runs under a role and a warehouse. The cell below sets your everyday role, makes sure your personal warehouse exists, and creates the `TT_LAB` schema in your own database — the safe sandbox where all of this lab's create/delete/recover work happens.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.TT_LAB;
USE SCHEMA {{my_db}}.TT_LAB;


---
## 1 · What Time Travel Gives You

[Time Travel](https://docs.snowflake.com/en/user-guide/data-time-travel) is Snowflake's built-in "undo" for your data. For a set **retention period**, Snowflake keeps the historical versions of every table, schema, and database — so you can look at, and recover, data as it was at an earlier point. There are **no backups to schedule or restore**; the history is already there.

Within the retention window you can:

| Do this | With |
|---|---|
| **Query** a table as it was at a past time | `AT(timestamp => ...)` / `AT(offset => -seconds)` |
| **Query** a table as it was *just before* a specific statement | `BEFORE(statement => '<query_id>')` |
| **Recover** an older state without duplicating storage | `CREATE TABLE ... CLONE ... AT/BEFORE(...)` |
| **Bring back** a dropped table, schema, or database | `UNDROP` |

**The retention window:**

| Edition | Default | Maximum |
|---|---|---|
| Standard | 1 day | 1 day |
| Enterprise and above | 1 day | 90 days (per object, via `DATA_RETENTION_TIME_IN_DAYS`) |

> :bulb: **Time Travel vs. Fail-safe.** Time Travel is *self-service* recovery within the retention window. After that window, data spends a short period in **Fail-safe** — but Fail-safe is recoverable only by Snowflake Support, not by you. Note that these protections vary by table type: **permanent** tables get Time Travel (up to 90 days on Enterprise+) *and* a 7-day Fail-safe, while **transient** and **temporary** tables have a maximum 1-day Time Travel window and **no Fail-safe** at all. The takeaway for an analyst: catch mistakes early, while they're still inside Time Travel, where recovery is one query away.

This lab uses the default **1-day** window, which is plenty — you'll recover data seconds after losing it.

---
## 2 · Build Your Analysis Table

You don't experiment on the shared source data — you pull it into your own sandbox first. The cell below copies `ORDERS` from the read-only `PLATPLUS_DB.ROASTERY_SALES` schema into a table **you own**, `TT_LAB.ORDERS_ANALYSIS`. This is your working copy — the one you'd point a dashboard or a revenue model at.

> :bulb: **Why a copy?** Time Travel works on *your* table's history. By owning `ORDERS_ANALYSIS`, you can safely break it and recover it without ever touching the shared source everyone else relies on.

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.TT_LAB.ORDERS_ANALYSIS AS
SELECT * FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;

SELECT COUNT(*) AS starting_rows FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS;


> :mag_right: **Review the output.** Note the `starting_rows` count — this is the healthy state you'll recover back to. Keep it in mind.

### Where the retention window is set. 🕳️

Recovery only works because your table sits inside a **Time Travel retention window**, controlled by one object parameter: `DATA_RETENTION_TIME_IN_DAYS`. It can be set at four levels — **account → database → schema → table** — and the **lowest (most specific) level wins**. A table with no value of its own inherits from its schema; the schema from its database; the database from the account default.

Two quick looks tell the whole story. First, `SHOW PARAMETERS` on the **account** reports the default value *and*, in the `level` column, *where* it's set — a **blank `level`** means it's inherited, not pinned. Then a single query against `INFORMATION_SCHEMA` reads the **effective retention** on your database, schema, and analysis table side by side:

In [ ]:
-- Account default — the value everything inherits unless overridden
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS' IN ACCOUNT;


In [ ]:
-- Effective retention (in days) on your database, schema, and analysis table — side by side.
-- SHOW PARAMETERS tells you WHERE a value is pinned; INFORMATION_SCHEMA tells you the effective days per object.
SELECT * FROM (
  SELECT 1 AS lvl, 'DATABASE' AS scope, DATABASE_NAME AS object_name, RETENTION_TIME AS retention_days
  FROM {{my_db}}.INFORMATION_SCHEMA.DATABASES
  WHERE DATABASE_NAME = CURRENT_DATABASE()
  UNION ALL
  SELECT 2, 'SCHEMA', SCHEMA_NAME, RETENTION_TIME
  FROM {{my_db}}.INFORMATION_SCHEMA.SCHEMATA
  WHERE SCHEMA_NAME = 'TT_LAB'
  UNION ALL
  SELECT 3, 'TABLE', TABLE_NAME, RETENTION_TIME
  FROM {{my_db}}.INFORMATION_SCHEMA.TABLES
  WHERE TABLE_SCHEMA = 'TT_LAB' AND TABLE_NAME = 'ORDERS_ANALYSIS'
) ORDER BY lvl;


> :mag_right: **Review both outputs.** The account `SHOW PARAMETERS` reports **`1`** with a **blank `level`** — the account is running on the default, with nothing pinned above it. The `INFORMATION_SCHEMA` query then confirms the same **1-day** window is effective at the database, schema, and table levels — each inherits from the account because nothing is pinned lower down. (Reading the `level` column: when it names a *higher* scope than the object you asked about, that object is **inheriting** from that scope; when it names the object's own scope, the value is **pinned** there.) Pin `DATA_RETENTION_TIME_IN_DAYS` on the schema or table and its `retention_days` here would rise — the lowest (most specific) setting wins.

> :bulb: **Why two tools?** `SHOW PARAMETERS` tells you *where* a value is set (the `level` column — inherited vs. pinned); `INFORMATION_SCHEMA` tells you the *effective* number of days per object and lets you list many objects at once. Reach for whichever answers the question in front of you.

> 📌 **Clock note.** "1 day" is a *rolling* 24-hour window measured from each change — not a calendar-day boundary. Delete at 3 PM today and that version stays recoverable until 3 PM tomorrow. Ask for a point more than 24 hours back with `AT(timestamp => ...)` and the query fails — the history has aged out.

### Bookmark this moment. 🔖

Before you make any changes, capture the current timestamp into a session variable. Think of it as a bookmark: *"the table was healthy at this instant."* You'll use it in Section 4 to query the table as it looked right now, with `AT(timestamp => ...)`.

In [ ]:
SET good_time = CURRENT_TIMESTAMP();
SELECT $good_time AS bookmarked_at;


---
## 3 · The Incident — A Bad Delete

Here's the mistake. You *meant* to remove a handful of test rows for a non-existent store, but you typed `store_id = 3` — a real, busy store. The `DELETE` commits instantly, and those orders vanish from your analysis table.

The cell also captures the query id of the delete into a session variable (`$bad_delete`) with [`LAST_QUERY_ID()`](https://docs.snowflake.com/en/sql-reference/functions/last_query_id). That id is the precise anchor you'll use to look back to the moment *just before* the damage.

> :point_right: **Your turn — fill in the blank.** The `DELETE` below is the scripted mistake — leave it exactly as-is. Your job is the **third line**: capture the id of the statement that just ran, so Section 4 can look back to the moment right before it. Which function returns the id of your most recent query?
>
> **Make sure you edit the third line before you execute the next cell.**

In [ ]:
-- Oops: a wrong WHERE removes a real store's orders
DELETE FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS WHERE store_id = 3;
SET bad_delete = ___;   -- capture the id of the query that just ran

<details>
<summary>📖 Reference solution</summary>

    SET bad_delete = LAST_QUERY_ID();

`LAST_QUERY_ID()` returns the id of the statement that ran immediately before it in this session — the DELETE. You anchor `BEFORE(statement => $bad_delete)` to it in Section 4.
</details>

> **Got a red error?** That's expected if you left `___` as-is. Don't use the Fix button — this is your puzzle to solve. The `DELETE` *already ran* (your data is gone), so re-running the cell won't help — `LAST_QUERY_ID()` would capture a second, empty delete instead of the original.
>
> Expand the **Stumbled?** section below — it has a recovery query that finds the original DELETE's query id from your session history.

In [ ]:
SELECT $bad_delete;

<details>
<summary>🛟 Stumbled? Were you able to retrieve the Query ID?</summary>

**What happened:** The `DELETE` in the cell above already ran — your data *is* deleted. But `$bad_delete` either doesn't exist (you left `___` as-is and got a syntax error) or holds a junk value (you replaced it with something other than `LAST_QUERY_ID()`). Either way, Section 4's `BEFORE(statement => $bad_delete)` can't find the right point in history.

**How to fix it:** Run the SQL below in a new cell. It searches your recent Query History for the actual DELETE statement and captures its id into `$bad_delete`:

```sql
SET bad_delete = (
  SELECT QUERY_ID FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY())
  WHERE QUERY_TEXT ILIKE 'DELETE%ORDERS_ANALYSIS%store_id%'
    AND QUERY_TYPE = 'DELETE'
  ORDER BY START_TIME DESC LIMIT 1
);
SELECT $bad_delete AS captured_query_id;
```

> **Ran the DELETE more than once?** Query History may point at the wrong run — your data was lost at the *first* delete, not the last. In that case, skip the query-id hunt and anchor your recovery to the `$good_time` bookmark you captured in Section 2 instead (`AT(timestamp => $good_time)`).
</details>

In [ ]:
SELECT COUNT(*) AS rows_after_delete FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS;

> :mag_right: **Review the output.** `rows_after_delete` is lower than `starting_rows` — store 3's orders are gone from the live table. **Don't panic.** The rows aren't destroyed; they're still sitting in Time Travel, and the next section proves it.

> 📌 **Lab shortcut vs. real life.** You captured `$bad_delete` in the very cell that caused the damage — tidy for a lab, but not how it happens on the job. In production you'd spot the problem later and hunt for the offending statement in [Query History](https://docs.snowflake.com/en/user-guide/ui-snowsight-activity#query-history), filtering by user, time window, and statement type. CoCo can do that hunt for you — later in this lab you'll ask it to find the `DELETE` in your query history.

> :warning: **Heads-up on session variables.** The `$good_time` and `$bad_delete` variables live in *this* session. If you restart the kernel or reconnect, re-run Sections 2 and 3 so they're set again before continuing — otherwise the `AT`/`BEFORE` cells below will error on an unset variable.

---
## 4 · Look Back With `AT` and `BEFORE`

Snowflake gives you two ways to see the past, and you'll use both. Neither changes anything — they're read-only windows into history.

- **`BEFORE(statement => ...)`** — the state *immediately before* a specific statement ran. Use it when you know which query caused the damage (you do: `$bad_delete`).
- **`AT(timestamp => ...)`** — the state *as of* a point in time. Use it when you have a known-good moment (you do: `$good_time`).

Both should return your original `starting_rows` count.

> :bulb: **They're interchangeable anchors.** `AT` and `BEFORE` each accept a **timestamp**, an **offset**, *or* a **statement id** — `BEFORE(timestamp => …)` and `AT(statement => …)` are equally valid. We pair them the most common way here (`BEFORE` with the statement that caused the damage, `AT` with a bookmarked time), but reach for whichever anchor you have.

> :point_right: **Your turn — fill in the blank.** You know which statement caused the damage (`$bad_delete`). Complete the Time Travel clause that shows the table's state *immediately before* that statement ran.

In [ ]:
-- The state right before the bad DELETE — fill in the Time Travel clause
SELECT COUNT(*) AS rows_before_delete
FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS ___(statement => $bad_delete);


<details>
<summary>📖 Reference solution</summary>

    SELECT COUNT(*) AS rows_before_delete
    FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS BEFORE(statement => $bad_delete);

`BEFORE(statement => …)` returns the data as it was just before the given query id executed — your original row count.
</details>

> :point_right: **Your turn — write this one yourself.** You just used `BEFORE(statement => …)`. Now use the other anchor: write a query returning the row count of `ORDERS_ANALYSIS` **as of the timestamp you bookmarked** in Section 2 (`$good_time`). It should match `starting_rows` too.

In [ ]:
-- Your turn: query the row count AT the bookmarked timestamp ($good_time)


<details>
<summary>📖 Reference solution</summary>

    SELECT COUNT(*) AS rows_at_bookmark
    FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS AT(timestamp => $good_time);

`AT(timestamp => …)` returns the data as of a point in time — here, the bookmark you set before the delete.
</details>

> :mag_right: **Review both outputs.** Each count matches `starting_rows` — proof the store-3 orders were never truly lost. Two different anchors (a statement and a timestamp), same healthy view of the table.

> :bulb: **The `AT(offset => ...)` variant.** You can also look back by a relative number of seconds — `AT(offset => -3600)` means "one hour ago." We use a bookmarked timestamp and a query id here because this table is only seconds old, so "an hour ago" would predate its creation. In day-to-day analysis, `AT(offset => -3600)` is a handy way to compare *"the numbers now vs. an hour ago"* — see the [`AT | BEFORE`](https://docs.snowflake.com/en/sql-reference/constructs/at-before) syntax reference.

---
## :robot_face: CoCo Assist

When something breaks, CoCo is your recovery partner — it can find the offending query, explain which Time Travel clause to use, and even draft the recovery SQL. Here's how you'd use it in the moment.

> :warning: **How to get the most from this section:**
>
> 1. **Read CoCo's response carefully** — the explanation is where the learning happens.
> 2. **CoCo is interactive** — ask follow-ups until it clicks ("What if I don't know the query id?").
> 3. **The prompts below are starting points** — your real learning comes from trying your own.

---

**Example 1: Explain AT vs BEFORE**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Explain the difference between AT and BEFORE in Snowflake Time Travel. When would I use AT(timestamp), AT(offset), and BEFORE(statement)? Give a one-line example of each.</pre>

---

**Example 2: Find the query that caused the damage**

You didn't capture the query id? CoCo can find it in your query history.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Look in my recent query history in INFORMATION_SCHEMA for any DELETE statements against ORDERS_ANALYSIS in the last hour. Show me the query id, the SQL text, and how many rows each removed.</pre>

*Then follow up with:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Using that query id, write the SQL to clone the table as it was just before that DELETE.</pre>

---

> :bulb: **These are just examples.** CoCo can help with any recovery question: "How much of my retention window is left?", "What's the difference between UNDROP and cloning?", "Did that DELETE also affect any other tables?" Ask in plain English and follow up until you're confident.

---
## 5 · Recover With a Point-in-Time Clone

Seeing the good data is one thing — getting it back is the goal. The safe pattern is: **clone the good state into a new table, verify it, then swap it in.** You never overwrite the live table until you're sure.

A [`CLONE`](https://docs.snowflake.com/en/sql-reference/sql/create-clone) is **zero-copy** — it's near-instant and consumes no extra storage until the clone and its source diverge. Cloning `BEFORE(statement => $bad_delete)` gives you a full copy of the table exactly as it was before the delete.

> :point_right: **Your turn — fill in the blank.** Recover the good state into a new table. Which keyword makes a **zero-copy** copy of another table (optionally at a past point via `BEFORE`/`AT`)?

In [ ]:
-- Zero-copy clone of the pre-delete state into a new table you can inspect
CREATE OR REPLACE TABLE {{my_db}}.TT_LAB.ORDERS_ANALYSIS_RECOVERED
___ {{my_db}}.TT_LAB.ORDERS_ANALYSIS BEFORE(statement => $bad_delete);   -- zero-copy copy keyword

SELECT COUNT(*) AS recovered_rows FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS_RECOVERED;


<details>
<summary>📖 Reference solution</summary>

    CREATE OR REPLACE TABLE {{my_db}}.TT_LAB.ORDERS_ANALYSIS_RECOVERED
    CLONE {{my_db}}.TT_LAB.ORDERS_ANALYSIS BEFORE(statement => $bad_delete);

`CLONE` makes a zero-copy copy; adding `BEFORE(statement => $bad_delete)` clones the table exactly as it was before the delete.
</details>

> :mag_right: **Review the output.** `recovered_rows` matches your original `starting_rows`. The recovered table is a safe, separate object — the live table is still in its damaged state, so you've lost nothing by inspecting first.

### Swap the recovered table in — atomically. 🔁

You've verified the clone is correct. Now make it the live table. Snowflake's [`ALTER TABLE … SWAP WITH`](https://docs.snowflake.com/en/sql-reference/sql/alter-table) exchanges two tables — their names, data, and metadata — in a **single atomic transaction**. There's never an instant where `ORDERS_ANALYSIS` doesn't exist, so any dashboard or query hitting it mid-swap is never disrupted. After the swap, the good data *is* `ORDERS_ANALYSIS`, and the old damaged rows land in `ORDERS_ANALYSIS_RECOVERED` — which you then drop.

In [ ]:
-- Atomic swap: ORDERS_ANALYSIS and the recovered clone trade places in ONE transaction,
-- so there is never a moment where ORDERS_ANALYSIS is missing (in-flight queries never break).
ALTER TABLE {{my_db}}.TT_LAB.ORDERS_ANALYSIS
  SWAP WITH {{my_db}}.TT_LAB.ORDERS_ANALYSIS_RECOVERED;

-- After the swap, the OLD damaged rows now sit in ORDERS_ANALYSIS_RECOVERED — drop that leftover.
DROP TABLE {{my_db}}.TT_LAB.ORDERS_ANALYSIS_RECOVERED;

SELECT COUNT(*) AS final_rows FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS;


> :mag_right: **Review the output.** `final_rows` matches `starting_rows` — store 3's orders are back in your analysis table. Investigate → look back → recover → verify → swap. That's the whole loop.

> :bulb: **Why `SWAP WITH` and not `DROP` + `RENAME`?** You *could* recover by dropping the damaged table and renaming the clone into its place — but between those two statements the name `ORDERS_ANALYSIS` briefly doesn't exist, and anything querying it in that window errors. `ALTER TABLE … SWAP WITH …` exchanges the two tables in a **single atomic transaction**, so the name always resolves and nothing in flight breaks. (Both tables must be owned by your role — they are, since you created both.)

> 🧑‍💻 **Why this matters to you as an analyst.** You just took your own table from damage to full recovery — diagnose, look back, restore, verify — with no ticket, no admin, and no rebuild. That's what makes a sandbox genuinely yours: within the retention window any mistake is reversible, so you can explore without fear of breaking something permanently.

### One more tool: `UNDROP`. ↩️

`CLONE … BEFORE` recovers *row-level* damage. But if you drop the **whole table** by accident, there's an even simpler rescue within the retention window: [`UNDROP`](https://docs.snowflake.com/en/sql-reference/sql/undrop-table).

First, drop the table and try to query it — you'll see the failure. Then, in the next cell, recover it with a single command.

> :point_right: **Step 1 — drop it.** Run the cell below. The `SELECT` will fail because the table no longer exists — that's the point. You'll fix it in the next cell.

In [ ]:
-- The accident: drop the whole table
DROP TABLE {{my_db}}.TT_LAB.ORDERS_ANALYSIS;

-- Try to read it — this WILL fail
SELECT COUNT(*) FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS;


> ⛔ **Expected error: table does not exist.** That's what a dropped table looks like to anyone querying it. Now recover it — fill in the blank below with the single command that restores a dropped table.

In [ ]:
___ TABLE {{my_db}}.TT_LAB.ORDERS_ANALYSIS;   -- one-statement recovery of a dropped table

SELECT COUNT(*) AS rows_after_undrop FROM {{my_db}}.TT_LAB.ORDERS_ANALYSIS;


<details>
<summary>📖 Reference solution</summary>

    UNDROP TABLE {{my_db}}.TT_LAB.ORDERS_ANALYSIS;

`UNDROP` reverses a `DROP` within the retention window — a table (or schema, or database) comes back exactly as it was, in one statement. Use it for a dropped *object*; use `AT`/`BEFORE` + `CLONE` for row-level damage inside a table.
</details>

> :mag_right: **Review the output.** `rows_after_undrop` matches your recovered row count — the table came back intact, in one statement. You saw the failure (table gone), then the fix (`UNDROP`) — that's the full cycle. Use `UNDROP` for a dropped *object*; use `AT`/`BEFORE` + `CLONE` for row-level damage inside a table that still exists.

> :bulb: **Beyond rows: column-level recovery.** The same `AT`/`BEFORE` + `CLONE` move also brings back *column-level* changes — clone the table from *before* an `ALTER TABLE … DROP COLUMN` and the dropped column (with its data) is present again in the clone, ready to copy back.

### See the whole story in your query history. 🧾

You just dropped and restored the table — and every statement you ran is recorded. `INFORMATION_SCHEMA.QUERY_HISTORY` lets you read your own recent activity back as a mini **audit trail**. Query it for anything that touched `ORDERS_ANALYSIS` and you'll see the incident and its recovery in order: the bad `DELETE`, the `CLONE`, the `SWAP`, the `DROP`, and the `UNDROP` — newest first.

In [ ]:
-- The audit trail: every recent statement that touched ORDERS_ANALYSIS, newest first.
-- INFORMATION_SCHEMA.QUERY_HISTORY is a table function — wrap it in TABLE(...).
SELECT QUERY_ID, QUERY_TYPE, EXECUTION_STATUS, START_TIME, QUERY_TEXT
FROM TABLE({{my_db}}.INFORMATION_SCHEMA.QUERY_HISTORY(RESULT_LIMIT => 200))
WHERE QUERY_TEXT ILIKE '%ORDERS_ANALYSIS%'
ORDER BY START_TIME DESC;


> :mag_right: **Review the output.** Read top-to-bottom (newest first) and you can retrace the whole incident. Your `DROP` shows as `QUERY_TYPE = DROP` — and here's the neat part: your `UNDROP` shows as **`RESTORE`**, Snowflake's internal name for the operation. The `DELETE`, `CREATE_TABLE_AS_SELECT`, and clone/swap statements are all here too. Reconstructing *what happened* is the first move in any recovery, and `QUERY_HISTORY` is where you do it.

> :bulb: **`QUERY_HISTORY` is scoped to you.** `INFORMATION_SCHEMA.QUERY_HISTORY` returns *your* recent queries (last 7 days). Admins use the account-wide [`SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY`](https://docs.snowflake.com/en/sql-reference/account-usage/query_history) view to see across all users — same idea, wider lens.

---
## Verify Your Work

The check below confirms your analysis table has been recovered to its original size — the same number of rows as the shared source. A green check means the recovery loop worked end to end.

In [ ]:
all_passed = True
try:
    src = session.sql('SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS').collect()[0][0]
    final = session.sql(f'SELECT COUNT(*) FROM {my_db}.TT_LAB.ORDERS_ANALYSIS').collect()[0][0]
    ok = final == src
    print(('✅' if ok else '⛔') + f' ORDERS_ANALYSIS recovered: {final} rows (source has {src})')
    if not ok:
        all_passed = False
        print('   Re-run Sections 4–5 — the recovered clone should match the source count.')
except Exception as e:
    print(f'⛔ Recovery check ERROR — {e}')
    all_passed = False

# The recovered clone should have been renamed IN (so the intermediate table is gone)
try:
    leftover = len(session.sql(f"SHOW TABLES LIKE 'ORDERS_ANALYSIS_RECOVERED' IN SCHEMA {my_db}.TT_LAB").collect())
    ok = leftover == 0
    print(('✅' if ok else '⛔') + f' Recovered clone swapped in (no leftover table): {"clean" if ok else "still present"}')
    if not ok:
        all_passed = False
except Exception as e:
    print(f'⛔ Swap check ERROR — {e}')
    all_passed = False

if all_passed:
    print('\n🎉 All checks passed — you recovered from a bad delete with Time Travel.')


---
### Check your understanding. 🧠
A few questions on this lab's key themes. Run each quiz cell below, choose an answer, and review the feedback.

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q1")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q2")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q3")

In [ ]:
from resources.knowledgecheck import data_analyst_quiz as quiz
quiz.display_question("q4")

---
### :snowflake: Key Takeaways

- Time Travel retains history for a **retention period** (1 day default; up to 90 on Enterprise+) — recover recent mistakes without a backup
- **`BEFORE(statement => query_id)`** sees the state just before a statement; **`AT(timestamp => ...)`** / **`AT(offset => -seconds)`** sees the state at a point in time
- A point-in-time **`CLONE`** is zero-copy — recover the good state into a new table, **verify it**, then swap it in atomically with **`ALTER TABLE … SWAP WITH …`** (no window where the live table is missing)
- Never overwrite the live table until you've confirmed the recovered copy is correct
- Dropped a whole table? Reach for **`UNDROP`** — the one-line rescue within the retention window
- **CoCo is your recovery partner** — it can find the offending query and draft the recovery SQL

**The Other Personas:**
- The **Data Engineer** notebook recovers a staging table from a bad pipeline **overwrite**
- The **Platform Admin** notebook recovers from an accidental **`DROP SCHEMA`** and tunes retention policy

> :bulb: Each persona lab is independent and uses its own objects — you can run them in any order.

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.